# Figure 2: Activity-based evaluation

**Paper:** BATTLE-AMP  
**Story:** Performance on binary AMP/non-AMP classification does not translate to indicating
peptides with antibacterial activity (MIC < 32 ug/ml). Regressors and HydrAMP-MIC are better.

**Four stacked panels (6.5 x 2 each):**
- **a** MCC dumbbell: AMP/non-AMP vs GeneralActivity per model
- **b** Scatter: AUROC (x) vs pAUROC at FPR<0.01 (y) on GeneralActivity
- **c** FPR comparison bars: AMP vs GeneralActivity
- **d** Precision@100 on GeneralActivity + SLAY Precision@100 placeholder

**Color conventions:**
- Model types: rose (classifiers), purple (HydrAMP-MIC), grey (regressors)
- Yellow and blue/green are reserved for active/inactive class labels across the paper

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from adjustText import adjust_text
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# Paths  (adjust to your local setup)
# ============================================================
CLF_FILE = Path("../results/aggregated/classification_results.tsv")
FIGURE_DIR = Path("../figures")
FIGURE_DIR.mkdir(exist_ok=True)

# ============================================================
# Figure constants
# ============================================================
PANEL_W = 6.5       # max width  (Briefings in Bioinformatics)
PANEL_H = 2.0       # per panel
TARGET_DPI = 600

# Model-type palette (yellow + blue/green reserved for active/inactive)
CLF_COLOR  = "#d95f6e"   # muted rose      -- classifiers
ACT_COLOR  = "#9467bd"   # muted purple    -- HydrAMP-MIC
REG_COLOR  = "#8c8c8c"   # medium grey     -- regressors

GRID_COLOR = "#dddddd"
FONTSIZE_TICK  = 6
FONTSIZE_LABEL = 7
FONTSIZE_PANEL = 10

In [ ]:
# ============================================================
# Models
# ============================================================
CLASSIFIERS = [
    "hydramp-amp-classifier", "ampscanner", "amplify",
    "sensexamp-classifier", "ampeppy", "ampredmfa", "mole-amp",
]
ACTIVITY_AWARE = ["hydramp-mic-classifier"]
REGRESSORS = [
    "mbc-attention", "ampredictor",
    "sensexamp-ecoli", "sensexamp-saureus",
    "deep-amp-cnn-gramneg", "deep-amp-cnn-grampos",
    "deep-amp-lstm-gramneg", "deep-amp-lstm-grampos",
    "apex-ecoli", "apex-saureus", "apex-min",
    "apex-abaumannii", "apex-paeruginosa", "apex-kpneumoniae",
]
ALL_MODELS = CLASSIFIERS + ACTIVITY_AWARE + REGRESSORS

MODEL_DISPLAY = {
    "hydramp-amp-classifier": "HydrAMP$_{AMP}$",
    "ampscanner": "AMP Scanner$_2$",
    "amplify": "AMPlify",
    "sensexamp-classifier": "sAMPpred$_{clf}$",
    "ampeppy": "amPEPpy",
    "ampredmfa": "AMPpred-MFA",
    "mole-amp": "MoLE-AMP",
    "hydramp-mic-classifier": "HydrAMP$_{MIC}$",
    "mbc-attention": "MBC-Attention",
    "ampredictor": "AMPredictor",
    "sensexamp-ecoli": "sAMPpred$_{EC}$",
    "sensexamp-saureus": "sAMPpred$_{SA}$",
    "deep-amp-cnn-gramneg": "Deep-AMP$_{CNN-}$",
    "deep-amp-cnn-grampos": "Deep-AMP$_{CNN+}$",
    "deep-amp-lstm-gramneg": "Deep-AMP$_{LSTM-}$",
    "deep-amp-lstm-grampos": "Deep-AMP$_{LSTM+}$",
    "apex-ecoli": "APEX$_{EC}$",
    "apex-saureus": "APEX$_{SA}$",
    "apex-min": "APEX$_{min}$",
    "apex-abaumannii": "APEX$_{AB}$",
    "apex-paeruginosa": "APEX$_{PA}$",
    "apex-kpneumoniae": "APEX$_{KP}$",
}


def model_color(m):
    if m in CLASSIFIERS:
        return CLF_COLOR
    elif m in ACTIVITY_AWARE:
        return ACT_COLOR
    return REG_COLOR


def short_name(m):
    return MODEL_DISPLAY.get(m, m)

In [ ]:
# ============================================================
# Load data
# ============================================================
df = pd.read_csv(CLF_FILE, sep="\t")
df = df[df["variant"] != "example-model"].copy()

amp = df[df["task"] == "amp"].set_index("variant")
ga  = df[df["task"] == "broad_activity"].set_index("variant")

ga_sorted = ga.sort_values("mcc", ascending=False)
model_order = [m for m in ga_sorted.index if m in ALL_MODELS]
n = len(model_order)
x = np.arange(n)

ga_base_rate = ga.iloc[0]["n_positive"] / (
    ga.iloc[0]["n_positive"] + ga.iloc[0]["n_negative"]
)
print(f"Models: {n}, GA base rate: {ga_base_rate:.3f}")

In [ ]:
# ============================================================
# Global style
# ============================================================
matplotlib.rcParams["font.family"] = "sans-serif"
matplotlib.rcParams["font.sans-serif"] = [
    "Helvetica", "Arial", "Liberation Sans", "DejaVu Sans",
]
matplotlib.rcParams["mathtext.default"] = "regular"
matplotlib.rcParams["axes.linewidth"] = 0.5
matplotlib.rcParams["xtick.major.width"] = 0.4
matplotlib.rcParams["ytick.major.width"] = 0.4


def style_ax(ax, ylabel=None, xlabel=None):
    ax.set_facecolor("white")
    ax.yaxis.grid(True, color=GRID_COLOR, linewidth=0.4)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=FONTSIZE_TICK, length=2)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=FONTSIZE_LABEL)
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=FONTSIZE_LABEL)


def set_model_xlabels(ax, order, rotation=90, fontsize=5):
    ax.set_xticks(np.arange(len(order)))
    labels = ax.set_xticklabels(
        [short_name(m) for m in order],
        rotation=rotation, ha="center", fontsize=fontsize,
    )
    for lbl, m in zip(labels, order):
        lbl.set_color(model_color(m))
    ax.set_xlim(-0.6, len(order) - 0.4)


def type_legend(ax, loc="upper right", ncol=3):
    handles = [
        Patch(facecolor=CLF_COLOR, edgecolor="none", label="Classifier"),
        Patch(facecolor=ACT_COLOR, edgecolor="none", label="HydrAMP$_{MIC}$"),
        Patch(facecolor=REG_COLOR, edgecolor="none", label="Regressor"),
    ]
    ax.legend(handles=handles, fontsize=5.5, loc=loc, frameon=True,
              facecolor="white", edgecolor="#cccccc", ncol=ncol,
              handlelength=1.0, handleheight=0.7, columnspacing=0.8)

In [ ]:
fig = plt.figure(figsize=(PANEL_W, PANEL_H * 4 + 0.6), dpi=TARGET_DPI,
                 facecolor="white")
gs = gridspec.GridSpec(4, 1, hspace=0.55,
                       left=0.12, right=0.97, top=0.975, bottom=0.04)

ax_a = fig.add_subplot(gs[0])
ax_b = fig.add_subplot(gs[1])
ax_c = fig.add_subplot(gs[2])
gs_d = gridspec.GridSpecFromSubplotSpec(
    1, 2, subplot_spec=gs[3], wspace=0.25, width_ratios=[3, 1],
)
ax_d1 = fig.add_subplot(gs_d[0])
ax_d2 = fig.add_subplot(gs_d[1])


# ================================================================
# Panel A: MCC dumbbell
# ================================================================
mcc_amp = [amp.loc[m, "mcc"] if m in amp.index else np.nan for m in model_order]
mcc_ga  = [ga.loc[m, "mcc"]  if m in ga.index  else np.nan for m in model_order]

for i, m in enumerate(model_order):
    c = model_color(m)
    ax_a.plot([i, i], [mcc_ga[i], mcc_amp[i]],
              color=c, linewidth=0.8, alpha=0.35, zorder=1)

ax_a.scatter(x, mcc_ga, s=22,
             c=[model_color(m) for m in model_order],
             edgecolors="none", zorder=3, clip_on=False)
ax_a.scatter(x, mcc_amp, s=22, facecolors="white",
             edgecolors=[model_color(m) for m in model_order],
             linewidths=0.8, zorder=2, clip_on=False)

set_model_xlabels(ax_a, model_order)
ax_a.set_ylim(-0.05, 0.88)
style_ax(ax_a, ylabel="MCC")

leg_handles = [
    Line2D([], [], marker='o', color='none', markerfacecolor='white',
           markeredgecolor='grey', markersize=5, label='AMP/non-AMP'),
    Line2D([], [], marker='o', color='none', markerfacecolor='grey',
           markeredgecolor='none', markersize=5, label='GeneralActivity'),
]
leg1 = ax_a.legend(
    handles=leg_handles, fontsize=5.5, loc="upper right",
    frameon=True, facecolor="white", edgecolor="#cccccc",
    handletextpad=0.3, borderpad=0.4,
)
ax_a.add_artist(leg1)
type_legend(ax_a, loc="upper center", ncol=3)


# ================================================================
# Panel B: Scatter AUROC vs pAUROC(0.01) on GA
# ================================================================
auroc_ga   = [ga.loc[m, "auroc"]      if m in ga.index else np.nan for m in model_order]
pauroc001  = [ga.loc[m, "pauroc_001"] if m in ga.index else np.nan for m in model_order]
colors_b   = [model_color(m) for m in model_order]

# Diagonal
ax_b.plot([0.493, 0.58], [0.493, 0.58], "--", color="#aaaaaa", linewidth=0.4,
          alpha=0.5, zorder=0, clip_on=True)
ax_b.text(0.57, 0.575, "no degradation", fontsize=4, color="#aaaaaa",
          rotation=28, ha="right", va="bottom", alpha=0.6)

# Random baselines
ax_b.axhline(0.5, color="#aaaaaa", linewidth=0.4, linestyle=":", alpha=0.5, zorder=0)
ax_b.axvline(0.5, color="#aaaaaa", linewidth=0.4, linestyle=":", alpha=0.5, zorder=0)

ax_b.scatter(auroc_ga, pauroc001, s=30, c=colors_b,
             edgecolors="white", linewidths=0.4, zorder=3)

# Label key models
to_label = {
    "mbc-attention", "sensexamp-ecoli", "apex-ecoli",
    "hydramp-mic-classifier", "hydramp-amp-classifier",
    "mole-amp", "ampredictor", "amplify",
}
texts_b = []
for i, m in enumerate(model_order):
    if m in to_label:
        t = ax_b.text(auroc_ga[i], pauroc001[i], short_name(m),
                      fontsize=4.5, color=model_color(m),
                      ha="center", va="center")
        texts_b.append(t)

adjust_text(
    texts_b, ax=ax_b,
    arrowprops=dict(arrowstyle="-", color="#aaaaaa", linewidth=0.2, alpha=0.4),
    expand=(2.0, 2.2),
    force_text=(1.0, 1.2),
    force_points=(0.6, 0.6),
    only_move={"text": "xy", "static": "xy", "explode": "xy", "pull": "xy"},
)

ax_b.set_xlim(0.48, 0.94)
ax_b.set_ylim(0.493, 0.58)
ax_b.xaxis.grid(True, color=GRID_COLOR, linewidth=0.4)
style_ax(ax_b, xlabel="AUROC (GeneralActivity)",
         ylabel="pAUROC (FPR<0.01)")
type_legend(ax_b, loc="upper left", ncol=1)


# ================================================================
# Panel C: FPR bars (AMP vs GeneralActivity)
# ================================================================
fpr_amp = [amp.loc[m, "fpr"] if m in amp.index else np.nan for m in model_order]
fpr_ga  = [ga.loc[m, "fpr"]  if m in ga.index  else np.nan for m in model_order]

bar_w = 0.38
ax_c.bar(x - bar_w / 2, fpr_amp, bar_w,
         color=[model_color(m) for m in model_order],
         alpha=0.35, edgecolor="#aaaaaa", linewidth=0.3)
ax_c.bar(x + bar_w / 2, fpr_ga, bar_w,
         color=[model_color(m) for m in model_order],
         alpha=1.0, edgecolor="#aaaaaa", linewidth=0.3)

set_model_xlabels(ax_c, model_order)
ax_c.set_ylim(0, 1.05)
style_ax(ax_c, ylabel="False Positive Rate")
ax_c.legend(
    [Patch(facecolor="#888888", alpha=0.35, edgecolor="#aaaaaa"),
     Patch(facecolor="#888888", alpha=1.0, edgecolor="#aaaaaa")],
    ["AMP/non-AMP", "GeneralActivity"],
    fontsize=5.5, loc="upper right", frameon=True,
    facecolor="white", edgecolor="#cccccc",
    handlelength=1.0, handleheight=0.7,
)


# ================================================================
# Panel D: Precision@100 on GA + SLAY placeholder
# ================================================================
prec100 = [
    ga.loc[m, "precision_at_k"] if m in ga.index else np.nan
    for m in model_order
]

ax_d1.bar(x, prec100, 0.65,
          color=[model_color(m) for m in model_order],
          edgecolor="#aaaaaa", linewidth=0.3)

# Random baseline = base rate
ax_d1.axhline(ga_base_rate, color="#aaaaaa", linewidth=0.5, linestyle=":", zorder=0)
ax_d1.text(n - 0.5, ga_base_rate + 0.005, f"base rate ({ga_base_rate:.2f})",
           fontsize=5, color="#888888", va="bottom", ha="right")

set_model_xlabels(ax_d1, model_order)
ax_d1.set_ylim(0, 1.05)
style_ax(ax_d1, ylabel="Precision@100\nGeneralActivity")

# SLAY placeholder
ax_d2.set_facecolor("white")
ax_d2.text(0.5, 0.55, "SLAY", ha="center", va="center",
           fontsize=8, color="#888888", fontweight="bold",
           transform=ax_d2.transAxes)
ax_d2.text(0.5, 0.38, "Precision@100\n(pending)", ha="center", va="center",
           fontsize=6, color="#aaaaaa", style="italic",
           transform=ax_d2.transAxes)
ax_d2.set_xticks([])
ax_d2.set_yticks([])
for spine in ax_d2.spines.values():
    spine.set_linestyle("--")
    spine.set_color("#bbbbbb")


# ================================================================
# Panel labels
# ================================================================
for label, ax in [("a", ax_a), ("b", ax_b), ("c", ax_c), ("d", ax_d1)]:
    pos = ax.get_position()
    fig.text(pos.x0 - 0.08, pos.y1 + 0.005, label,
             fontsize=FONTSIZE_PANEL, fontweight="bold",
             va="bottom", ha="left")


# ================================================================
# Save
# ================================================================
out_pdf = FIGURE_DIR / "figure2_activity.pdf"
out_png = FIGURE_DIR / "figure2_activity.png"
fig.savefig(str(out_pdf), bbox_inches="tight", dpi=TARGET_DPI, facecolor="white")
fig.savefig(str(out_png), bbox_inches="tight", dpi=TARGET_DPI, facecolor="white")
print(f"Saved {out_pdf} and {out_png}")
plt.show()


## Alt text

**Figure 2.** Activity-based benchmarking of 22 AMP prediction models, comparing performance
on binary AMP/non-AMP classification against the clinically grounded GeneralActivity task
(MIC < 32 ug/ml). Models are color-coded by type: classifiers (rose), HydrAMP-MIC (purple),
regressors (grey).  
**(a)** Dumbbell chart of Matthews Correlation Coefficient. Open circles show AMP/non-AMP
performance; filled circles show GeneralActivity. Vertical lines connect paired values.
Classifiers show dramatic drops; regressors maintain or improve.  
**(b)** Scatter of AUROC versus pAUROC at FPR<0.01 on GeneralActivity. Despite AUROC spanning
0.50 to 0.87, pAUROC compresses to 0.50 to 0.56. MBC-Attention and sAMPpred-EC separate
upward; most classifiers cluster near the random baseline.  
**(c)** False positive rate comparison. Light bars: AMP/non-AMP; full-opacity bars:
GeneralActivity. Most classifiers exceed FPR 0.78 on GeneralActivity.  
**(d)** Precision@100 on GeneralActivity (base rate 0.78). Most models achieve near-perfect
precision in this positive-enriched setting; the SLAY dataset (placeholder at right) will
test screening under realistic imbalanced conditions.

## TODO
- [ ] Wire SLAY Precision@100 results into panel d
- [ ] Verify model ordering matches Figure 3
- [ ] Propagate color palette changes to Figure 3 notebook
- [ ] Fine-tune panel b label positions after final data